# Mortalité Départementale 2024

Charger fichier multi-onglets

https://data.drees.solidarites-sante.gouv.fr/api/datasets/1.0/601_indicateurs-de-contexte/attachments/c03_isd_taux_de_mortalite_...


In [8]:
import pandas as pd
from pathlib import Path

# On essaie d'utiliser __file__, sinon on prend le dossier actuel
try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()

ROOT = BASE_DIR.parent
DATA_DIR = ROOT / "data"

%ls ../data/raw/territoires

print(DATA_DIR)

ADE-COG_4-0_GPKG_LAMB93_FXX-ED2025-01-01.gpkg*
C03-ISD_Taux_de_mortalite.xlsx
ZE2020_au_01-01-2025.xlsx
densite_20240101.xlsx
georef-france-zone-emploi-2020.geojson
iris_france.gpkg*
/Users/jean-jacques/code/jjchabutDataCRM/sante-territoires/data


In [9]:
file_path = DATA_DIR / 'raw/territoires/C03-ISD_Taux_de_mortalite.xlsx'

# Voir les onglets disponibles
xl = pd.ExcelFile(file_path)
print(f"Onglets disponibles : {xl.sheet_names}")

Onglets disponibles : ['DEP', 'Documentation']


In [10]:
# Charger l'onglet Dep
df = pd.read_excel(
    file_path,
    sheet_name='DEP',
    skiprows=5,
    #dtype={'Code commune': str}  # Garder les zéros initiaux
)

In [11]:
df.head()

,Code,Département,Taux brut de mortalité en 2024 (en ‰),Taux brut de mortalité des femmes en 2024 (en ‰),Taux brut de mortalité des hommes en 2024 (en ‰),Taux de mortalité standard. des 0-64 ans en 2024 (prématuré) (en ‰),Taux de mortalité standard. des 65 ans ou plus en 2024 (en ‰),Taux de mortalité infantile (pour 1000 enfants nés vivants entre 2022 et 2024) (en ‰),Nombre de décès domiciliés en 2024
0,01,Ain,7.9,7.8,7.9,1.4,36.4,3.6,5283.0
1,02,Aisne,11.0,10.7,11.3,2.2,41.8,4.0,5822.0
2,03,Allier,13.5,13.4,13.7,2.1,37.5,2.8,4545.0
3,04,Alpes-de-Haute-Provence,11.8,11.5,12.2,1.8,36.0,3.2,1982.0
4,05,Hautes-Alpes,10.6,10.2,10.9,1.7,31.1,2.5,1519.0


In [12]:
df.isna().sum()

Code                                                                                     8
Département                                                                              2
Taux brut de mortalité en 2024 (en ‰)                                                    2
Taux brut de mortalité des femmes en 2024 (en ‰)                                         2
Taux brut de mortalité des hommes en 2024 (en ‰)                                         2
Taux de mortalité standard. des 0-64 ans en 2024 (prématuré) (en ‰)                      2
Taux de mortalité standard. des 65 ans ou plus en 2024 (en ‰)                            2
Taux de mortalité infantile (pour 1000 enfants nés vivants entre 2022 et 2024) (en ‰)    2
Nombre de décès domiciliés en 2024                                                       2
dtype: int64

In [21]:
df_nuls = df[df['Département'].isnull()]
df_nuls

,Code,Département,Taux brut de mortalité en 2024 (en ‰),Taux brut de mortalité des femmes en 2024 (en ‰),Taux brut de mortalité des hommes en 2024 (en ‰),Taux de mortalité standard. des 0-64 ans en 2024 (prématuré) (en ‰),Taux de mortalité standard. des 65 ans ou plus en 2024 (en ‰),Taux de mortalité infantile (pour 1000 enfants nés vivants entre 2022 et 2024) (en ‰),Nombre de décès domiciliés en 2024
111,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
112,"Source : Insee, État civil, Estimations de pop...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
df_clean = df[~df['Département'].isnull()]

In [23]:
df_clean.columns

Index(['Code', 'Département', 'Taux brut de mortalité en 2024 (en ‰)',
       'Taux brut de mortalité des femmes en 2024 (en ‰)',
       'Taux brut de mortalité des hommes en 2024 (en ‰)',
       'Taux de mortalité standard. des 0-64 ans en 2024 (prématuré) (en ‰)',
       'Taux de mortalité standard. des 65 ans ou plus en 2024 (en ‰)',
       'Taux de mortalité infantile (pour 1000 enfants nés vivants entre 2022 et 2024) (en ‰)',
       'Nombre de décès domiciliés en 2024'],
      dtype='object')

In [24]:
new_names = {
    'Taux brut de mortalité en 2024 (en ‰)': 'taux_brut',
    'Taux brut de mortalité des femmes en 2024 (en ‰)': 'taux_femmes',
    'Taux brut de mortalité des hommes en 2024 (en ‰)': 'taux_hommes',
    'Taux de mortalité standard. des 0-64 ans en 2024 (prématuré) (en ‰)': 'taux_prematuré',
    'Taux de mortalité standard. des 65 ans ou plus en 2024 (en ‰)': 'taux_65plus',
    'Taux de mortalité infantile (pour 1000 enfants nés vivants entre 2022 et 2024) (en ‰)': 'taux_infantile',
    'Nombre de décès domiciliés en 2024': 'nb_deces'
}

df_clean = df_clean.rename(columns=new_names)

In [25]:
df_clean.isna().sum()

Code              7
Département       0
taux_brut         0
taux_femmes       0
taux_hommes       0
taux_prematuré    0
taux_65plus       0
taux_infantile    0
nb_deces          0
dtype: int64

In [31]:
file_path = DATA_DIR / 'extracted/raw_dept_taux_mortalité_2024.parquet'
df_clean.to_parquet(file_path)

file_path = DATA_DIR / 'extracted/raw_dept_taux_mortalité_2024.csv'
df_clean.to_csv(file_path)


In [32]:
ls -lh ../data/extracted/

total 9840
-rw-r--r--  1 jean-jacques  staff   700K Feb 24 15:08 communes_31_geom_simplified.parquet
-rw-r--r--  1 jean-jacques  staff   704K Feb 24 15:54 communes_31_geom_simplified_geo.parquet
-rw-r--r--  1 jean-jacques  staff   3.3M Feb 20 22:28 communes_densite.csv
-rw-r--r--  1 jean-jacques  staff    11K Mar  4 12:20 dept_taux_mortalité_2024.parquet
-rw-r--r--@ 1 jean-jacques  staff   5.8K Feb 20 23:56 mortalite_departements.csv
-rw-r--r--  1 jean-jacques  staff   5.6K Mar  4 12:25 raw_dept_taux_mortalité_2024.csv
-rw-r--r--  1 jean-jacques  staff    11K Mar  4 12:25 raw_dept_taux_mortalité_2024.parquet
